# Katip Mask Mini · train in the browser-size class (15-30 MB)

**What this does.** Katip Mask Base (110 MB, our best model) teaches three small models trained from scratch
(knowledge distillation). The best one on the *dev* sets that is also the smallest becomes **Katip Mask Mini**.
Test sets are scored once at the end, so nothing is tuned on them.

| | |
|---|---|
| Time | ~5 min smoke test, then ~1.5-2.5 h on a free T4 |
| Needs | GPU runtime. Nothing to upload: code from GitHub, data from public sources, teacher from Hugging Face |
| Output | `katip-mask-mini.zip` (ONNX int8 for the browser + PyTorch weights) and a results table |
| Licence | CC BY-SA 4.0 (the training text includes Turkish Wiki NER), like Nano and Base |

**How to run:** Runtime > Run all. If the smoke test cell ends with a results line, the full run will work.

In [ ]:
# 1 · Check the GPU, get the code. Run this cell first. (Takes ~1 min.)
import os, subprocess, sys
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "NONE. In Colab: Runtime > Change runtime type > T4 GPU. In Kaggle: Settings > Accelerator > GPU T4.")
if not os.path.exists("katip"):
    subprocess.run("git clone -q --depth 1 -b legal-study https://github.com/fikrifirat/katip", shell=True, check=True)
os.chdir("katip")
subprocess.run(f"{sys.executable} -m pip install -q regex tokenizers onnx onnxruntime faker huggingface_hub 'transformers>=4.44' accelerate", shell=True, check=True)
print(subprocess.run("git log --oneline -1", shell=True, capture_output=True, text=True).stdout)

In [ ]:
# Optional · Hugging Face token, only needed to upload at the end.
# Colab: the key icon on the left > add a secret named HF_TOKEN (write access) and allow this notebook.
# Kaggle: Add-ons > Secrets > HF_TOKEN. Nothing is hard-coded here.
HF_TOKEN = None
try:
    from google.colab import userdata; HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient; HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
print("HF token:", "found" if HF_TOKEN else "none (fine: you can download the zip instead)")

## 2 · Smoke test (5 min): every step on a little data, results thrown away

In [ ]:
!cd lab && MODE=smoke python mini/train_mini.py 2>&1 | grep -v Warning | tail -5

## 3 · Full run (~2 h)
Candidates: `mini-a` 384 hidden × 4 layers, `mini-b` 384 × 6, `mini-c` 512 × 4, each with a 32k vocabulary learned
on real + synthetic text. Keep the tab open (Colab disconnects idle tabs after ~90 min without interaction on the free tier;
Kaggle runs up to 12 h with "Save Version > Save & Run All").

In [ ]:
!cd lab && MODE=full python mini/train_mini.py 2>&1 | grep -v Warning | tail -25

## 4 · Results

In [ ]:
import json, pandas as pd
r = json.load(open("lab/mini/runs/katip-mask-mini_results.json"))
display(pd.DataFrame(r["candidates"]))
w = r["winner"]; print(f"winner: {w['arch']}  {w['int8_mb']} MB  trust {w['trust']}")
t = r["test"]["wiki test"]
print(f"Wikipedia names (test): recall {t['recall']:.1%}  precision {t['precision']:.1%}   (Nano ~51%, Base ~90%)")
print(f"synthetic blind recall {r['test']['blind']['recall']:.1%}   false names on name-free text: {r['test']['negatives']['false']}")

In [ ]:
# Save the result: a zip you can download (left panel > Files) and, if you set HF_TOKEN, a PRIVATE upload to Hugging Face.
# Make it public later on the model page (Settings > Visibility) after checking the scores.
import shutil, os
z = shutil.make_archive("katip-mask-mini", "zip", "lab/mini/runs")
print("zip:", z, round(os.path.getsize(z) / 1e6, 1), "MB")
try:
    from google.colab import files; files.download(z)
except Exception:
    pass
UPLOAD = False   # set True to upload
if UPLOAD and HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo("f1k/katip-mask-mini", private=True, exist_ok=True)
    api.upload_folder(folder_path="lab/mini/runs", repo_id="f1k/katip-mask-mini", commit_message="trained in a notebook (katip-mask-mini)")
    print("uploaded to https://huggingface.co/f1k/katip-mask-mini (private)")

**Next:** send the results line (or the zip) back in the Katip thread. We add Mini to KVKK-Bench and the browser demo from there.